In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import window, col, count, sum, to_timestamp, to_date, current_timestamp

In [0]:
schemaLocation = '/Volumes/capgemini_academy/landing/metadata/_schema/events'
checkpointLocation = '/Volumes/capgemini_academy/landing/metadata/_checkpoint/events'
volume = '/Volumes/capgemini_academy/landing/files/events'
bronze_table = 'capgemini_academy.bronze.events_stream'

stream_df = (spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'json')
    .option('cloudFiles.schemaLocation', schemaLocation)
    .load(volume)
)

query = (stream_df.writeStream
    .format('delta')
    .option('checkpointLocation', checkpointLocation)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM capgemini_academy.bronze.events_stream

In [0]:
events = spark.read.table(bronze_table)

agg = (events
        .withWatermark('event_ts', '10 minutes')
        .groupBy(window('event_ts', '5 minutes'), col('event_type'))
        .agg(count('*').alias('events_count'))
        )

agg.display()

In [0]:
catalog = 'capgemini_academy'
schema_volume = 'landing'
schema_bronze = 'bronze'
schema_silver = 'silver'

source_path = f'/Volumes/{catalog}/{schema_volume}/files/events'
checkpoint_path = f'/Volumes/{catalog}/{schema_volume}/metadata/_checkpoint/events'
target_table = f'{catalog}.{schema_bronze}.events_stream'

print(source_path)
print(checkpoint_path)
print(target_table)

In [0]:
schema_events = StructType([
  StructField('event_id', StringType()),
  StructField('pedido_id', StringType()),
  StructField('cliente_id', StringType()),
  StructField('produto_id', StringType()),
  StructField('event_type', StringType()),
  StructField('valor_total', DoubleType()),
  StructField('event_ts', StringType())
])

raw_stream = (spark.readStream
    .schema(schema_events)
    .json(source_path)
)

dbutils.fs.rm(checkpoint_path + '_preview1', True)

print(raw_stream)

# Para visualizar streaming sem agregação, use writeStream com trigger availableNow
query = (raw_stream.writeStream
    .format('memory')
    .queryName('raw_stream_preview')
    .outputMode('append')
    .option('checkpointLocation', checkpoint_path + '_preview1')
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

spark.sql('SELECT * FROM raw_stream_preview LIMIT 100').display()

In [0]:
events_stream = (raw_stream
  .withColumn('event_ts', to_timestamp('event_ts'))
  .withColumn('event_date', to_date('event_ts'))
  .withColumn('ingestion_ts', current_timestamp())
  .filter(col('event_id').isNotNull())
)

In [0]:
# OPÇÃO 2: Usando checkpoint diferente + schema evolution
checkpoint_v2 = checkpoint_path + '_v2'

query = (events_stream.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', checkpoint_v2)
    .option('mergeSchema', 'true')  # Permite adicionar novas colunas
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
agg_result = (events_stream
    .withWatermark('event_ts', '10 minutes')
    .groupBy(window('event_ts', '5 minutes'), 'event_type')
    .agg(count('*').alias('events_count'), sum('valor_total').alias('total_value')
  )
)

dbutils.fs.rm(checkpoint_path + '_preview2', True)

# Para visualizar streaming sem agregação, use writeStream com trigger availableNow
query = (agg_result.writeStream
    .format('memory')
    .queryName('agg_result_preview')
    .outputMode('append')
    .option('checkpointLocation', checkpoint_path + '_preview2')
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

spark.sql('SELECT * FROM agg_result_preview LIMIT 100').display()

In [0]:
agg_table = f'{catalog}.{schema_silver}.events_agg_5min'
agg_checkpoint = f'/Volumes/{catalog}/{schema_volume}/metadata/_checkpoint/events_agg_5min'

agg_query = (agg_result.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', agg_checkpoint)
    .trigger(availableNow=True)
    .toTable(agg_table)
)

agg_query.awaitTermination()

In [0]:
%sql
SELECT * FROM capgemini_academy.silver.events_agg_5min

In [0]:
# Execute apenas se quiser reiniciar o exercício do zero
RESET_LAB = True

if RESET_LAB:
    for t in [bronze_table, agg_table]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")

    # Remove checkpoints e tabelas para recriar do zero
    dbutils.fs.rm(checkpoint_path, True)
    dbutils.fs.rm(agg_checkpoint, True)
    dbutils.fs.rm(checkpoint_path + '_preview1', True)
    dbutils.fs.rm(checkpoint_path + '_preview2', True)
    dbutils.fs.rm(checkpoint_v2, True)
    dbutils.fs.rm(schemaLocation, True)
    print("Laboratório reiniciado.")
else:
    print("RESET_LAB = False. Nenhum objeto foi removido.")